# Hindcast 0008-01 Tropospheric Wave / Ozone Tests

Purpose: reproduce the `Hindcast_vertical_analysis.ipynb` O3 RMSE vs winter-wave-activity figure with the cleaned Hindcast products, then test which EP-flux wave component and which tropospheric signals matter for `0008-01`.

This notebook writes only under:

```text
/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/TEST_TROPOS/outputs/0008-01
```

Data are read from `/mnt/soclim0/public_data/weiji`. Run with the `jimnew` environment.

In [ ]:
from __future__ import annotations

import glob
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from scipy.stats import linregress, pearsonr

plt.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
})

CASE = "0008-01"
REF_YEAR = 8

REPO_ROOT = Path("/home/weiji/restart_exam/code_cleaned")
TEST_ROOT = REPO_ROOT / "Hindcast_experiment" / "TEST_TROPOS"
OUT_ROOT = TEST_ROOT / "outputs" / CASE
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"
CACHE_DIR = OUT_ROOT / "cache"
for d in [FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/mnt/soclim0/public_data/weiji")
HINDCAST_ROOT = DATA_ROOT / "Hindcast"
BWCN_ROOT = DATA_ROOT / "BWCN"
B2000_ROOT = DATA_ROOT / "B2000WCN001002_timefixed"

CASE_ROOT = HINDCAST_ROOT / CASE

# Main windows. The EP window matches the old Hindcast_vertical_analysis scatter.
O3_END = (5, 30)
EP_WINDOW = ((1, 20), (2, 10))
EARLY_AO_WINDOW = ((1, 1), (1, 19))
Z300_WINDOW = EP_WINDOW

LAT_EP = (40.0, 80.0)
LAT_POLAR = (60.0, 90.0)
LAT_Z300 = (20.0, 90.0)
PLEV_EP_PA = 10000.0
PLEV_U_PA = 1000.0
PLEV_Z300_PA = 30000.0

WAVES = ["all_waves", "wave1", "wave2", "wave_rest"]
WAVE_LABELS = {
    "all_waves": "All waves",
    "wave1": "Wave 1",
    "wave2": "Wave 2",
    "wave_rest": "Wave rest",
}

# Expensive sections cache their results. Keep True for the full requested test.
RUN_Z300_DIAGNOSTICS = True
BUILD_U60N10_IF_MISSING = True
CLIM_MAX_BWCN_YEARS_FOR_Z300 = None  # None = all BWCN Z3 years available.

print(f"Output root: {OUT_ROOT}")
print(f"Case root exists: {CASE_ROOT.exists()} -> {CASE_ROOT}")

In [ ]:
# -----------------------------
# Shared helpers
# -----------------------------

def member_short_id(member) -> str:
    text = str(member)
    m = re.search(r"\.(\d{3})\.cam\.h3", text)
    if m:
        return m.group(1)
    m = re.search(r"\.(\d{3})\.", text)
    return m.group(1) if m else text


def date_parts(date_values):
    arr = np.asarray(date_values, dtype=np.int64)
    year = arr // 10000
    mmdd = arr % 10000
    month = mmdd // 100
    day = mmdd % 100
    return year, month, day


def date_mask(date_values, start=(1, 1), end=(5, 30), year: Optional[int] = None):
    yy, mm, dd = date_parts(date_values)
    start_key = start[0] * 100 + start[1]
    end_key = end[0] * 100 + end[1]
    key = mm * 100 + dd
    mask = (key >= start_key) & (key <= end_key)
    if year is not None:
        mask = mask & (yy == int(year))
    return mask


def one_dim_date(ds_or_da) -> np.ndarray:
    date = ds_or_da["date"]
    if "member" in date.dims:
        date = date.isel(member=0)
    return np.asarray(date.values, dtype=np.int64)


def assign_member_short_coord(da: xr.DataArray) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    mids = [member_short_id(v) for v in da["member"].values]
    return da.assign_coords(member_short=("member", mids))


def select_latband(da: xr.DataArray, lat_range: Tuple[float, float], lat_name="lat") -> xr.DataArray:
    lat = da[lat_name]
    descending = float(lat.values[0]) > float(lat.values[-1])
    lo, hi = lat_range
    return da.sel({lat_name: slice(hi, lo) if descending else slice(lo, hi)})


def coslat_mean(da: xr.DataArray, lat_range: Optional[Tuple[float, float]] = None, lat_name="lat") -> xr.DataArray:
    if lat_range is not None:
        da = select_latband(da, lat_range, lat_name=lat_name)
    weights = np.cos(np.deg2rad(da[lat_name])).clip(0, 1)
    return da.weighted(weights.fillna(0)).mean(lat_name, skipna=True)


def finite_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    r, p = pearsonr(x[mask], y[mask])
    return float(r), float(p)


def scatter_fit(ax, df, xcol, ycol, title, xlabel=None, ylabel=None, color="tab:blue", annotate_members=True):
    sub = df[[xcol, ycol, "member"]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(sub[xcol], sub[ycol], s=62, color=color, edgecolor="black", linewidth=0.5, alpha=0.88)
    if annotate_members:
        for _, row in sub.iterrows():
            ax.text(row[xcol], row[ycol], str(row["member"]), fontsize=7, alpha=0.65)
    if len(sub) >= 3:
        fit = linregress(sub[xcol].values, sub[ycol].values)
        xx = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(xx, fit.slope * xx + fit.intercept, color="crimson", ls="--", lw=1.8)
        ax.text(
            0.03, 0.97,
            f"R = {fit.rvalue:.3f}\nP = {fit.pvalue:.2e}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.84, edgecolor="0.7"),
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel or xcol)
    ax.set_ylabel(ylabel or ycol)
    return ax


def savefig(fig, name):
    png = FIG_DIR / f"{name}.png"
    pdf = FIG_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved: {png}")
    return png, pdf


def mmdd_label(date_values):
    _, mm, dd = date_parts(date_values)
    return np.array([f"{int(m):02d}-{int(d):02d}" for m, d in zip(mm, dd)])


def month_ticks(date_values):
    _, mm, dd = date_parts(date_values)
    positions, labels = [], []
    names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun"}
    seen = set()
    for i, (m, d) in enumerate(zip(mm, dd)):
        if int(d) == 1 and int(m) not in seen:
            positions.append(i)
            labels.append(names.get(int(m), str(int(m))))
            seen.add(int(m))
    return positions, labels


def standardize_1d(y):
    y = np.asarray(y, dtype=float)
    return (y - np.nanmean(y)) / np.nanstd(y)

In [ ]:
# -----------------------------
# Data loaders for the cleaned Hindcast products
# -----------------------------

def load_hindcast_o3(case=CASE, pressure_tag="30_70hPa"):
    path = HINDCAST_ROOT / case / "partial_O3" / f"{case}_partial_O3_all_ranges_members.nc"
    if not path.exists():
        raise FileNotFoundError(f"Missing cleaned partial O3 product: {path}")
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        da = assign_member_short_coord(ds[var]).load()
        date = one_dim_date(ds)
    mask = date_mask(date, start=(1, 1), end=O3_END)
    da = da.isel(lead_time=mask)
    date = date[mask]
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_bwcn_ref_o3(year=REF_YEAR, pressure_tag="30_70hPa"):
    path = BWCN_ROOT / "partial_O3" / "BWCN_partial_O3_all_ranges.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=(1, 1), end=O3_END, year=year)
        da = ds[var].isel(time=mask).load()
        date = date[mask]
    da = da.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return da, date


def load_epflux_wave(case=CASE, wave="all_waves", plev_pa=PLEV_EP_PA, lat_range=LAT_EP):
    path = HINDCAST_ROOT / case / "EPflux_daily_ubar" / wave / f"EPFLUX_{wave}_{case}_members_time_plev_lat.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        ep2 = assign_member_short_coord(ds["ep2"])
        ep2_100 = ep2.sel(plev=plev_pa, method="nearest")
        ep2_100 = coslat_mean(ep2_100, lat_range=lat_range)
        # Use -ep2 so positive means upward wave activity, matching the old Fz scatter convention.
        out = (-ep2_100).load()
    return out


def ep_window_mean(ep_da: xr.DataArray, date_values, start_end=EP_WINDOW):
    start, end = start_end
    mask = date_mask(date_values, start=start, end=end)
    return ep_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def load_all_wave_metrics(case=CASE, date_values=None, start_end=EP_WINDOW):
    if date_values is None:
        _, date_values = load_hindcast_o3(case)
    rows = []
    series = {}
    for wave in WAVES:
        da = load_epflux_wave(case, wave=wave)
        da = da.isel(lead_time=slice(0, len(date_values)))
        metric = ep_window_mean(da, date_values, start_end=start_end)
        series[wave] = da.assign_coords(date=("lead_time", date_values[: da.sizes["lead_time"]]))
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    return pd.DataFrame(rows), series


def compute_rmse_table(o3_hind: xr.DataArray, date_h, o3_ref: xr.DataArray, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30"):
    mh = date_mask(date_h, start=start, end=end)
    mr = date_mask(date_ref, start=start, end=end, year=REF_YEAR)
    h = o3_hind.isel(lead_time=mh)
    r = o3_ref.isel(lead_time=mr)
    n = min(h.sizes["lead_time"], r.sizes["lead_time"])
    h = h.isel(lead_time=slice(0, n))
    r = r.isel(lead_time=slice(0, n))
    diff = h - r
    rmse = np.sqrt((diff ** 2).mean("lead_time", skipna=True))
    return pd.DataFrame({
        "member": [str(v) for v in rmse["member_short"].values],
        "RMSE_DU": rmse.values.astype(float),
        "rmse_window": label,
        "n_days": n,
    }).sort_values("RMSE_DU").reset_index(drop=True)


def merge_rmse_ep(rmse_df, ep_metric_df):
    wide = ep_metric_df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    return rmse_df.merge(wide, on="member", how="left")

In [ ]:
# -----------------------------
# Product sanity check for 0008-01
# -----------------------------
expected = {
    "partial_O3": CASE_ROOT / "partial_O3" / f"{CASE}_partial_O3_all_ranges_members.nc",
    "EP all": CASE_ROOT / "EPflux_daily_ubar" / "all_waves" / f"EPFLUX_all_waves_{CASE}_members_time_plev_lat.nc",
    "EP wave1": CASE_ROOT / "EPflux_daily_ubar" / "wave1" / f"EPFLUX_wave1_{CASE}_members_time_plev_lat.nc",
    "EP wave2": CASE_ROOT / "EPflux_daily_ubar" / "wave2" / f"EPFLUX_wave2_{CASE}_members_time_plev_lat.nc",
    "EP rest": CASE_ROOT / "EPflux_daily_ubar" / "wave_rest" / f"EPFLUX_wave_rest_{CASE}_members_time_plev_lat.nc",
    "FWD": CASE_ROOT / "final_warming_date" / f"{CASE}_FWD_plev_member.nc",
    "AO/NAM projection": CASE_ROOT / "NAM_B2000WCN_projection" / f"{CASE}_AO_NAM_B2000WCN_projection_members.nc",
}
for name, path in expected.items():
    print(f"{name:18s}: {path.exists()}  {path}")

with xr.open_dataset(expected["partial_O3"], decode_times=False) as ds:
    print("\npartial_O3 dims:", dict(ds.sizes))
    print("partial_O3 vars:", list(ds.data_vars))
with xr.open_dataset(expected["EP all"], decode_times=False) as ds:
    print("\nEP all dims:", dict(ds.sizes))
    print("EP attrs:", {k: ds.attrs.get(k) for k in ["method", "do_ubar", "use_omega_w_correction"]})

In [ ]:
# -----------------------------
# Reproduce O3 RMSE vs winter wave activity with cleaned data
# -----------------------------
o3_h, date_h = load_hindcast_o3(CASE)
o3_ref, date_ref = load_bwcn_ref_o3(REF_YEAR)

ep_metric_df, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)
rmse_all = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30")
rmse_ep_all = merge_rmse_ep(rmse_all, ep_metric_df)
rmse_ep_all.to_csv(TABLE_DIR / f"{CASE}_rmse_epflux_wave_metrics_Jan20_Feb10.csv", index=False)

print(rmse_ep_all.head())
print("\nRMSE range:", rmse_ep_all["RMSE_DU"].min(), rmse_ep_all["RMSE_DU"].max())
print("Saved table:", TABLE_DIR / f"{CASE}_rmse_epflux_wave_metrics_Jan20_Feb10.csv")

fig, ax = plt.subplots(figsize=(7.4, 5.4), constrained_layout=True)
scatter_fit(
    ax,
    rmse_ep_all,
    "EP100_all_waves",
    "RMSE_DU",
    title="Cleaned data: O3 RMSE vs early winter all-wave EP flux",
    xlabel="Mean 100 hPa -EP2, 40-80N, Jan20-Feb10",
    ylabel="O3 RMSE, 60-90N 30-70 hPa, Jan1-May30 (DU)",
    color="dodgerblue",
)
savefig(fig, f"{CASE}_reproduce_RMSE_vs_allwave_EPFlux")
plt.show()

In [ ]:
# -----------------------------
# Which wave component matters most?
# -----------------------------
summary_rows = []
for wave in WAVES:
    xcol = f"EP100_{wave}"
    r, p = finite_corr(rmse_ep_all[xcol], rmse_ep_all["RMSE_DU"])
    summary_rows.append({"wave": wave, "R_EP100_vs_RMSE": r, "P": p})
wave_corr = pd.DataFrame(summary_rows)
wave_corr.to_csv(TABLE_DIR / f"{CASE}_wave_component_correlations.csv", index=False)
print(wave_corr)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for ax, wave in zip(axes.ravel(), WAVES):
    scatter_fit(
        ax,
        rmse_ep_all,
        f"EP100_{wave}",
        "RMSE_DU",
        title=WAVE_LABELS[wave],
        xlabel=f"{WAVE_LABELS[wave]} mean 100 hPa -EP2",
        ylabel="O3 RMSE (DU)",
        color={"all_waves":"black", "wave1":"tab:blue", "wave2":"tab:orange", "wave_rest":"tab:green"}[wave],
        annotate_members=False,
    )
savefig(fig, f"{CASE}_RMSE_vs_EPFlux_by_wave")
plt.show()

# Spread timing: does EP-flux spread lead ozone spread?
x = np.arange(len(date_h))
fig, ax = plt.subplots(figsize=(11.5, 5.2), constrained_layout=True)
# O3 ensemble spread.
o3_spread = o3_h.std("member", skipna=True).values
ax.plot(x, standardize_1d(o3_spread), color="crimson", lw=2.5, label="O3 column spread")
for wave in WAVES:
    da = ep_series[wave].isel(lead_time=slice(0, len(date_h)))
    spread = da.std("member", skipna=True).values
    ax.plot(x, standardize_1d(spread), lw=1.7, label=f"{WAVE_LABELS[wave]} EP spread")
mask_ep = date_mask(date_h, start=EP_WINDOW[0], end=EP_WINDOW[1])
if mask_ep.any():
    ax.axvspan(np.where(mask_ep)[0][0], np.where(mask_ep)[0][-1], color="0.85", alpha=0.5, label="EP window")
positions, labels = month_ticks(date_h)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel("Standardized ensemble spread")
ax.set_title("Spread timing: O3 vs EP-flux wave components")
ax.legend(ncol=2, fontsize=9)
savefig(fig, f"{CASE}_spread_timing_O3_EPFlux_waves")
plt.show()

In [ ]:
# -----------------------------
# RMSE-window sensitivity: whole window vs Jan20+ vs Feb20+
# -----------------------------
rmse_windows = [
    ("RMSE_Jan01_May30", (1, 1), "Jan1-May30"),
    ("RMSE_Jan20_May30", (1, 20), "Jan20-May30"),
    ("RMSE_Feb20_May30", (2, 20), "Feb20-May30"),
]
merged_frames = []
for col, start, label in rmse_windows:
    df = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=start, end=O3_END, label=label)
    df = df.rename(columns={"RMSE_DU": col})[["member", col, "n_days"]]
    merged_frames.append(df)
rmse_multi = merged_frames[0]
for df in merged_frames[1:]:
    rmse_multi = rmse_multi.merge(df, on="member", how="inner", suffixes=("", "_y"))
rmse_multi = rmse_multi.merge(rmse_ep_all[["member"] + [f"EP100_{w}" for w in WAVES]], on="member", how="left")
rmse_multi.to_csv(TABLE_DIR / f"{CASE}_rmse_window_sensitivity.csv", index=False)
print(rmse_multi.head())

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.9), constrained_layout=True, sharex=True)
for ax, (col, _, label) in zip(axes, rmse_windows):
    dfp = rmse_multi.rename(columns={col: "RMSE_window"})
    scatter_fit(
        ax, dfp, "EP100_all_waves", "RMSE_window",
        title=label,
        xlabel="Mean 100 hPa all-wave -EP2, Jan20-Feb10",
        ylabel="O3 RMSE (DU)",
        color="tab:purple",
        annotate_members=False,
    )
savefig(fig, f"{CASE}_RMSE_window_sensitivity_vs_allwave_EPFlux")
plt.show()

In [ ]:
# -----------------------------
# AO test: use B2000WCN-projected AO/NAM if available.
# -----------------------------

def load_projected_ao(case=CASE):
    new_path = HINDCAST_ROOT / case / "NAM_B2000WCN_projection" / f"{case}_AO_NAM_B2000WCN_projection_members.nc"
    if not new_path.exists():
        print(f"Missing projected AO/NAM product: {new_path}")
        print("The legacy NAM/*.nc file lacks a reliable member axis for this scatter, so AO tests are skipped until the projected product is generated.")
        return None, None
    with xr.open_dataset(new_path, decode_times=False) as ds:
        ao = assign_member_short_coord(ds["AO_Index"]).load()
        date = one_dim_date(ds)
    ao = ao.assign_coords(date=("lead_time", date))
    return ao, date


ao, date_ao = load_projected_ao(CASE)
if ao is not None:
    rows = []
    early_mask = date_mask(date_ao, start=EARLY_AO_WINDOW[0], end=EARLY_AO_WINDOW[1])
    ep_mask = date_mask(date_ao, start=EP_WINDOW[0], end=EP_WINDOW[1])
    ao_early = ao.isel(lead_time=early_mask).mean("lead_time", skipna=True)
    ao_ep = ao.isel(lead_time=ep_mask).mean("lead_time", skipna=True)
    for i, mid in enumerate(ao["member_short"].values):
        rows.append({"member": str(mid), "AO_early_Jan01_19": float(ao_early.isel(member=i)), "AO_Jan20_Feb10": float(ao_ep.isel(member=i))})
    ao_df = pd.DataFrame(rows)
    ao_join = rmse_ep_all.merge(ao_df, on="member", how="left")
    ao_join.to_csv(TABLE_DIR / f"{CASE}_AO_EP_RMSE_metrics.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    scatter_fit(axes[0], ao_join, "AO_early_Jan01_19", "RMSE_DU", "Early AO vs O3 RMSE", "AO mean Jan1-19", "O3 RMSE (DU)", "tab:cyan")
    scatter_fit(axes[1], ao_join, "AO_Jan20_Feb10", "EP100_all_waves", "AO vs concurrent all-wave EP", "AO mean Jan20-Feb10", "100 hPa all-wave -EP2", "tab:cyan")
    savefig(fig, f"{CASE}_AO_tests_RMSE_EPFlux")
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    for ax, wave in zip(axes.ravel(), WAVES):
        scatter_fit(ax, ao_join, "AO_Jan20_Feb10", f"EP100_{wave}", f"AO vs {WAVE_LABELS[wave]}", "AO mean Jan20-Feb10", f"{WAVE_LABELS[wave]} -EP2", "tab:cyan", annotate_members=False)
    savefig(fig, f"{CASE}_AO_vs_wave_components")
    plt.show()

In [ ]:
# -----------------------------
# Z300 tropospheric diagnostics
# -----------------------------

def interp_profile_logp(da_var: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: float) -> xr.DataArray:
    target = np.array([float(p_tgt_pa)], dtype=float)
    def _interp_col(vcol, pcol):
        vcol = np.asarray(vcol, dtype=float)
        pcol = np.asarray(pcol, dtype=float)
        mask = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if mask.sum() < 2:
            return np.array([np.nan], dtype=float)
        p = pcol[mask]
        v = vcol[mask]
        idx = np.argsort(p)
        return np.interp(np.log(target), np.log(p[idx]), v[idx], left=np.nan, right=np.nan)
    out = xr.apply_ufunc(
        _interp_col,
        da_var,
        p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="allowed",
        output_dtypes=[float],
    )
    return out.assign_coords(plev=("plev", target)).isel(plev=0, drop=True)


def z300_from_file(path: Path, start_end=Z300_WINDOW, lat_range=LAT_Z300) -> xr.DataArray:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask)
        ds = ds.sel(lat=slice(lat_range[0], lat_range[1]))
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        z = interp_profile_logp(ds["Z3"].transpose("time", "lat", "lon", "lev"), p_mid.transpose("time", "lat", "lon", "lev"), PLEV_Z300_PA)
        z_mean = z.mean("time", skipna=True).load()
    z_mean.name = "Z300"
    z_mean.attrs.update({"units": "m", "plev_pa": PLEV_Z300_PA, "window": str(start_end)})
    return z_mean


def build_z300_hindcast_cache(case=CASE, overwrite=False):
    out = CACHE_DIR / f"Z300_{case}_members_Cwindow.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    files = sorted((HINDCAST_ROOT / case / "Z3").glob("*.Z3.nc"))
    if not files:
        raise FileNotFoundError(f"No Z3 files for {case}")
    das, mids = [], []
    for f in files:
        mid = member_short_id(f.name)
        print("Z300", case, mid)
        das.append(z300_from_file(f))
        mids.append(mid)
    da = xr.concat(das, dim=pd.Index(mids, name="member"))
    da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return da


def build_z300_bwcn_ref_cache(year=REF_YEAR, overwrite=False):
    out = CACHE_DIR / f"Z300_BWCN_{year:04d}_Cwindow.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    f = BWCN_ROOT / "Z3" / f"BWCN.cam.h3.{year:04d}.Z3.nc"
    da = z300_from_file(f)
    da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return da


def build_z300_jan_stationary_target(overwrite=False, max_years=CLIM_MAX_BWCN_YEARS_FOR_Z300):
    out = CACHE_DIR / "Z300_BWCN_Jan_stationary_wave_target.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    files = sorted((BWCN_ROOT / "Z3").glob("BWCN.cam.h3.*.Z3.nc"))
    if max_years is not None:
        files = files[: int(max_years)]
    das = []
    for f in files:
        print("Z300 Jan clim", f.name)
        da = z300_from_file(f, start_end=((1, 1), (1, 31)), lat_range=LAT_Z300)
        das.append(da)
    clim = xr.concat(das, dim="year_index").mean("year_index", skipna=True)
    target = clim - clim.mean("lon", skipna=True)
    target.name = "Z300_Jan_stationary_wave_target"
    target.to_netcdf(out)
    print(f"Saved cache: {out}")
    return target


def weighted_pattern_corr(a: xr.DataArray, b: xr.DataArray, lat_range=LAT_Z300, remove_zonal_mean=True):
    a, b = xr.align(select_latband(a, lat_range), select_latband(b, lat_range), join="inner")
    if remove_zonal_mean:
        a = a - a.mean("lon", skipna=True)
        b = b - b.mean("lon", skipna=True)
    w = np.sqrt(np.cos(np.deg2rad(a["lat"])).clip(0, 1))
    aw = (a * w).values.ravel()
    bw = (b * w).values.ravel()
    mask = np.isfinite(aw) & np.isfinite(bw)
    if mask.sum() < 10:
        return np.nan
    return float(np.corrcoef(aw[mask], bw[mask])[0, 1])


def weighted_projection(a: xr.DataArray, target: xr.DataArray, lat_range=LAT_Z300):
    a, target = xr.align(select_latband(a, lat_range), select_latband(target, lat_range), join="inner")
    a = a - a.mean("lon", skipna=True)
    target = target - target.mean("lon", skipna=True)
    w = np.cos(np.deg2rad(a["lat"])).clip(0, 1)
    num = (a * target * w).sum(skipna=True)
    den = (target * target * w).sum(skipna=True)
    return float((num / den).values)


if RUN_Z300_DIAGNOSTICS:
    z300_members = build_z300_hindcast_cache(CASE)
    z300_ref = build_z300_bwcn_ref_cache(REF_YEAR)
    z300_target = build_z300_jan_stationary_target()

    z_rows = []
    for mid in z300_members["member"].values:
        z = z300_members.sel(member=mid)
        z_rows.append({
            "member": str(mid),
            "Z300_AC_to_BWCN0008": weighted_pattern_corr(z, z300_ref),
            "Z300_stationary_corr": weighted_pattern_corr(z, z300_target),
            "Z300_stationary_projection": weighted_projection(z, z300_target),
        })
    z300_df = pd.DataFrame(z_rows)
    z300_join = rmse_ep_all.merge(z300_df, on="member", how="left")
    z300_join.to_csv(TABLE_DIR / f"{CASE}_Z300_tropospheric_metrics.csv", index=False)
    print(z300_join.head())

    fig, axes = plt.subplots(2, 2, figsize=(12.5, 9.2), constrained_layout=True)
    scatter_fit(axes[0, 0], z300_join, "Z300_AC_to_BWCN0008", "EP100_all_waves", "Z300 AC vs all-wave EP", "Z300 AC to BWCN0008", "All-wave -EP2", "tab:olive")
    scatter_fit(axes[0, 1], z300_join, "Z300_stationary_corr", "EP100_all_waves", "Stationary-wave closeness vs all-wave EP", "Z300 stationary corr", "All-wave -EP2", "tab:olive")
    scatter_fit(axes[1, 0], z300_join, "Z300_stationary_projection", "EP100_wave1", "Stationary-wave projection vs wave1 EP", "Projection onto Jan stationary wave", "Wave1 -EP2", "tab:olive")
    scatter_fit(axes[1, 1], z300_join, "Z300_stationary_projection", "RMSE_DU", "Stationary-wave projection vs O3 RMSE", "Projection onto Jan stationary wave", "O3 RMSE (DU)", "tab:olive")
    savefig(fig, f"{CASE}_Z300_tropospheric_signal_tests")
    plt.show()
else:
    print("RUN_Z300_DIAGNOSTICS is False; skipping Z300 cache build and scatter plots.")

In [ ]:
# -----------------------------
# Poster-style evolution plots from cleaned products.
# O3 uses new partial_O3. U60N10 uses raw Hindcast U and a local cache.
# -----------------------------

def load_case_o3_for_evolution(case):
    da, date = load_hindcast_o3(case)
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_ref_o3_curve(year=REF_YEAR):
    da, date = load_bwcn_ref_o3(year)
    return da, date


def load_b2000_o3_climatology():
    path = B2000_ROOT / "partial_O3" / "B2000WCN_partial_O3_all_ranges.nc"
    if not path.exists():
        print(f"Missing B2000 O3 climatology source: {path}")
        return None
    with xr.open_dataset(path, decode_times=False) as ds:
        da = ds["O3_partial_60_90N_30_70hPa"]
        date = np.asarray(ds["date"].values, dtype=np.int64)
        _, mm, dd = date_parts(date)
        mmdd = np.array([m * 100 + d for m, d in zip(mm, dd)])
        da = da.assign_coords(mmdd=("time", mmdd))
        clim = da.groupby("mmdd").mean("time", skipna=True).load()
    # Select Jan1-May30 by mmdd integer.
    all_mmdd = clim["mmdd"].values
    mask = (all_mmdd >= 101) & (all_mmdd <= 530)
    clim = clim.isel(mmdd=mask).rename({"mmdd": "lead_time"}).assign_coords(lead_time=np.arange(mask.sum()))
    return clim


def u60n10_from_file(path: Path, start_end=((1, 1), (5, 30))) -> Tuple[xr.DataArray, np.ndarray]:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask).sel(lat=60.0, method="nearest")
        date = date[mask]
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        u = interp_profile_logp(ds["U"].transpose("time", "lon", "lev"), p_mid.transpose("time", "lon", "lev"), PLEV_U_PA)
        u_zm = u.mean("lon", skipna=True).load()
    u_zm = u_zm.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return u_zm, date


def build_u60n10_case_cache(case, overwrite=False):
    out = CACHE_DIR / f"U60N10_{case}_members.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        date = np.asarray(da["date"].values, dtype=np.int64)
        return da, date
    files = sorted((HINDCAST_ROOT / case / "U").glob("*.U.nc"))
    if not files:
        raise FileNotFoundError(f"No U files for {case}")
    das, mids = [], []
    date0 = None
    for f in files:
        mid = member_short_id(f.name)
        print("U60N10", case, mid)
        da, date = u60n10_from_file(f)
        das.append(da)
        mids.append(mid)
        date0 = date
    out_da = xr.concat(das, dim=pd.Index(mids, name="member"))
    out_da.name = "U60N10"
    out_da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return out_da, date0


def build_u60n10_ref_cache(year=REF_YEAR, overwrite=False):
    out = CACHE_DIR / f"U60N10_BWCN_{year:04d}.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        return da, np.asarray(da["date"].values, dtype=np.int64)
    f = BWCN_ROOT / "U" / f"BWCN.cam.h3.{year:04d}.U.nc"
    da, date = u60n10_from_file(f)
    da.name = "U60N10"
    da.to_netcdf(out)
    return da, date


def plot_evolution(ref_data, ref_date, clim_data, experiments, ylabel, ylim, title, name, xlim=(0, 150)):
    fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
    xs = np.arange(len(ref_date))
    ax.plot(xs, ref_data.values, color="black", lw=3.5, label="Reference", zorder=10)
    if clim_data is not None:
        ax.plot(np.arange(clim_data.sizes[clim_data.dims[0]]), clim_data.values, color="black", ls=":", lw=3.2, label="Climatology", zorder=9)
    for exp in experiments:
        data, offset, color, label = exp["data"], exp["offset"], exp["color"], exp["label"]
        if data is None:
            continue
        total = data.sizes["lead_time"]
        x = np.arange(offset, offset + total)
        keep = (x >= xlim[0]) & (x <= xlim[1])
        x = x[keep]
        sub = data.isel(lead_time=keep)
        if "member" in sub.dims:
            ens = sub.transpose("member", "lead_time")
            mean = ens.mean("member", skipna=True)
            std = ens.std("member", skipna=True)
            for i in range(ens.sizes["member"]):
                ax.plot(x, ens.isel(member=i).values, color=color, alpha=0.18, lw=1.0)
            ax.fill_between(x, (mean - std).values, (mean + std).values, color=color, alpha=0.25)
            ax.plot(x, mean.values, color=color, lw=3.5, label=label)
        else:
            ax.plot(x, sub.values, color=color, lw=3.5, label=label)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(title, fontsize=15)
    ax.set_xticks([0, 31, 59, 90, 120, 151])
    ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
    ax.legend(fontsize=10)
    savefig(fig, name)
    plt.show()


def load_diag_for_case(diag, case):
    if diag == "O3":
        return load_case_o3_for_evolution(case)[0]
    if diag == "U60N10":
        da, _ = build_u60n10_case_cache(case)
        return da
    raise ValueError(diag)


evolution_specs = [
    {
        "name": "INT3D_vs_CLIM3D_0008_FebInit",
        "experiments": [
            ("0008-02", "H-INT-3D", 31, "forestgreen"),
            ("0008-02_NOCOUPL", "H-CLIM-3D", 31, "royalblue"),
        ],
    },
    {
        "name": "JAN_INITIAL_ONLY_0008",
        "experiments": [
            ("0008-01", "H-JanInit", 0, "darkorange"),
        ],
    },
]

for diag in ["O3", "U60N10"]:
    if diag == "O3":
        ref, ref_date = load_ref_o3_curve(REF_YEAR)
        clim = load_b2000_o3_climatology()
        ylabel, ylim = "Partial O3 column, 30-70 hPa (DU)", (60, 150)
    else:
        ref, ref_date = build_u60n10_ref_cache(REF_YEAR)
        old_clim_path = Path("/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3/cache/U60N10/U60N10_B2000WCN_climatology.nc")
        clim = xr.open_dataarray(old_clim_path).load() if old_clim_path.exists() else None
        ylabel, ylim = "Zonal-mean U at 60N, 10 hPa (m s-1)", (-50, 90)
    for spec in evolution_specs:
        experiments = []
        for case, label, offset, color in spec["experiments"]:
            try:
                data = load_diag_for_case(diag, case)
            except FileNotFoundError as exc:
                print(f"Skip {diag} {case}: {exc}")
                data = None
            experiments.append({"data": data, "label": label, "offset": offset, "color": color})
        plot_evolution(
            ref, ref_date, clim, experiments, ylabel, ylim,
            title=f"{diag}: {spec['name']}",
            name=f"{diag}_{spec['name']}",
        )

## Notes For Interpretation

- The RMSE metric is the cleaned `partial_O3` product: polar-cap `60-90N`, `30-70 hPa`, compared against BWCN year 0008 over the selected date window.
- EP-flux wave activity is `-ep2` at `100 hPa`, area-averaged over `40-80N`, so positive values follow the old upward-wave-activity convention.
- The AO block requires the new `NAM_B2000WCN_projection` output. The old `NAM/0008-01_Daily_AO_Index.nc` has no reliable member axis for member-level scatter diagnostics.
- Z300 diagnostics define two tropospheric predictors: pattern correlation to the BWCN year-0008 C-window Z300 pattern, and projection/correlation onto the January climatological stationary-wave target.